In [26]:
from itertools import product
from collections import Counter
import os
from random import randint
import re
import pickle as pkl

from catboost import CatBoostClassifier
import nltk
from nltk.corpus import stopwords
import nltk.stem as st
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline

from prepare_text_class import TextPrepareClass
from utilities import seedEverything

In [2]:
rand_seed = 141592
seedEverything(rand_seed)

## Задаю переменные

In [3]:
PATH_DATA = os.path.join('.', 'data')
PATH_SUBM = os.path.join('.', 'submissions')
PATH_MODL = os.path.join('.', 'models')

In [4]:
PHONES = r'\b(samsung|galaxy|xiaomi|iphon|redmi|note|honor|huawei|apple|' +\
          'nokia|meizu|google|самсунг|айфон|mi|lenovo|lg|redme|asus|vivo|' +\
          'zte|helio|mediatek|oppo|htc|pixel|xperia|fly|realme|zenfone|' +\
          'alcatel|blade|philips|touch|lumia|oneplus|motorola|inoi|red|neo|' +\
          'moto|panasonic|band|honnor|bbk|vertex|lafleur|xiomi|редми|хонор|' +\
          'ноки|хуаве|мейзу|асус|галакси|иной|гэлакси|хонор)(|[a-zа-я]+)' #\b

## Загрузка, очистка и преобразование данных

In [5]:
df = pd.read_csv(os.path.join(PATH_DATA, 'ru_train.csv'), )
df.shape

(4850, 5)

In [6]:
%%time
textprepare = TextPrepareClass(PHONES, stopwords.words('russian'))
df['text_cl'] = df.review.map(textprepare.clean_all)
df['target']  = df.rating.apply(lambda x: 0 if int(x) >= 4 else 1)

df = df.sample(frac=1).reset_index(drop=True)
df.sample(3)[['review', 'rating', 'link', 'target', 'text_cl']]

CPU times: user 51.4 s, sys: 145 ms, total: 51.5 s
Wall time: 51.5 s


,review,rating,link,target,text_cl
3389,"Не зря Apple считают одной из лучших фирм, пот...",5.0,https://irecommend.ru/content/moya-prelest-282,0,не зря счита одн лучш фирм техник действительн...
1668,Купила это телефон примерно год назад за 5000 ...,1.0,https://irecommend.ru/content/polnyi-uzhas-foto,1,куп эт телефон примерн год назад рубл снача не...
1883,Вся моя семья фанаты сотовых телефонов фирмы s...,5.0,https://irecommend.ru/content/moya-lyubov-k-sm...,0,вся сем фанат сотов телефон фирм одн врем пере...


In [7]:
df['target'].value_counts(normalize=True)

target
0    0.548454
1    0.451546
Name: proportion, dtype: float64

## Создание и сохранение модели и токенайзера для инференса через   
## onnx + docker + flask

Очищенные отзывы векторизую через tf-idf.   
Полученные векторы в CatBoostClassifier для подбора параметров.

In [8]:
grid_pipe = Pipeline([('vector', TfidfVectorizer(analyzer = 'char_wb')),
                     ('model', CatBoostClassifier(verbose=1000)),
                     ])

params_pipe = {'vector__max_features': [None],       # 256, 512, 768, 
               'vector__ngram_range': [(2, 3), (3, 4), (2, 4)],
               'vector__min_df': [0.1, 0.2, 0.3],     # 0.5
               'vector__max_df': [0.75, 0.85, 0.95],
               'model__iterations': [1000],  # 500
               'model__learning_rate': [.03, .06],
               'model__depth': [3, 6],
               'model__l2_leaf_reg': [2, 3],
               }

In [9]:
%%time
clf = GridSearchCV(grid_pipe, params_pipe,
                   n_jobs=-1, cv=5,
                   scoring = ['roc_auc', 'accuracy'],
                   # refit='roc_auc',
                   verbose=0,
                  )
clf.fit(df.text_cl, df.target)
clf.best_score_, clf.best_params_

0:	learn: 0.6862095	total: 362ms	remaining: 3m
400:	learn: 0.3251034	total: 1m 41s	remaining: 25s
499:	learn: 0.2903121	total: 1m 57s	remaining: 0us
0:	learn: 0.6859219	total: 124ms	remaining: 1m 1s
400:	learn: 0.3438701	total: 55.2s	remaining: 13.6s
499:	learn: 0.3094430	total: 1m 8s	remaining: 0us
0:	learn: 0.6880571	total: 102ms	remaining: 50.8s
400:	learn: 0.3439396	total: 38.4s	remaining: 9.47s
499:	learn: 0.3104196	total: 47.8s	remaining: 0us
0:	learn: 0.6871961	total: 195ms	remaining: 1m 37s
400:	learn: 0.3409425	total: 52.4s	remaining: 12.9s
499:	learn: 0.3065194	total: 1m 3s	remaining: 0us
0:	learn: 0.6870424	total: 68.7ms	remaining: 34.3s
400:	learn: 0.3526665	total: 31.4s	remaining: 7.74s
499:	learn: 0.3176685	total: 37.9s	remaining: 0us
0:	learn: 0.6876830	total: 115ms	remaining: 57.6s
400:	learn: 0.3494129	total: 35.4s	remaining: 8.73s
499:	learn: 0.3150601	total: 42.2s	remaining: 0us
0:	learn: 0.6852289	total: 194ms	remaining: 1m 36s
400:	learn: 0.3328747	total: 1m 1s	rem

/home/v010ch/anaconda3/envs/sentiment_phone/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


0:	learn: 0.6806062	total: 123ms	remaining: 2m 3s
400:	learn: 0.2294994	total: 49s	remaining: 1m 13s
800:	learn: 0.1213163	total: 1m 41s	remaining: 25.2s
999:	learn: 0.0911968	total: 2m 4s	remaining: 0us
0:	learn: 0.6789699	total: 443ms	remaining: 7m 22s
400:	learn: 0.2197048	total: 1m 7s	remaining: 1m 40s
800:	learn: 0.1140126	total: 2m 21s	remaining: 35.1s
999:	learn: 0.0847953	total: 2m 54s	remaining: 0us
0:	learn: 0.6797407	total: 163ms	remaining: 2m 42s
400:	learn: 0.2426693	total: 40.8s	remaining: 1m
800:	learn: 0.1350765	total: 1m 20s	remaining: 19.9s
999:	learn: 0.1041777	total: 1m 39s	remaining: 0us
0:	learn: 0.6815983	total: 124ms	remaining: 2m 3s
400:	learn: 0.2343658	total: 51.1s	remaining: 1m 16s
800:	learn: 0.1246777	total: 1m 44s	remaining: 26s
999:	learn: 0.0933926	total: 2m 7s	remaining: 0us
0:	learn: 0.6852777	total: 327ms	remaining: 5m 26s
400:	learn: 0.3227640	total: 1m 36s	remaining: 2m 23s
800:	learn: 0.2142075	total: 3m 16s	remaining: 48.7s
999:	learn: 0.1803944	

(np.float64(0.9480130119820098),
 {'model__depth': 6,
  'model__iterations': 1000,
  'model__l2_leaf_reg': 3,
  'model__learning_rate': 0.06,
  'vector__max_df': 0.85,
  'vector__max_features': None,
  'vector__min_df': 0.1,
  'vector__ngram_range': (2, 4)})

Сохраняю результаты на всех возможных комбинациях заданных параметров   
для дальнейшего анализа

In [12]:
res = pd.DataFrame(clf.cv_results_['params'])
res['mean_test_roc_auc'] = clf.cv_results_['mean_test_roc_auc']
res['mean_test_accuracy'] = clf.cv_results_['mean_test_accuracy']
res['mean_sum'] = (clf.cv_results_['mean_test_accuracy'] +\
                   clf.cv_results_['mean_test_roc_auc']
                   )/2

res.to_csv(os.path.join(PATH_DATA, 'reslt_grid1_catboost.csv'), index=False)
#res.to_csv(os.path.join(PATH_DATA, 'reslt_grid1_vect_catboost.csv'), index=False)

Определяю на какой итерации необходимо остановиться, что бы не переобучаться

In [63]:
%%time
iterations = []
scores = []
for idx in range(20):
    print(f'iter {idx}')
    X_train, X_test, y_train, y_test = train_test_split(
        df.text_cl, df.target, test_size=0.3, random_state=randint(0, 10000),
    )

    vectorizer = TfidfVectorizer(analyzer='char_wb',
                             ngram_range=clf.best_params_['vector__ngram_range'],
                             max_df=clf.best_params_['vector__max_df'],
                             min_df=clf.best_params_['vector__min_df'],
                            )
    train = vectorizer.fit_transform(X_train)
    test = vectorizer.transform(X_test)

    model = CatBoostClassifier(iterations=5000,
                           learning_rate=clf.best_params_['model__learning_rate'],
                           depth=clf.best_params_['model__depth'],
                           l2_leaf_reg=clf.best_params_['model__l2_leaf_reg'],
                           loss_function='Logloss',
                           eval_metric='AUC',
                           verbose=200,
                           )
    model.fit(train, y_train,
              eval_set=(test, y_test),
              early_stopping_rounds=100,
              # plot=True,
             )
    iterations.append(model.best_iteration_)
    scores.append(model.best_score_['validation']['AUC'])

iter 0
0:	test: 0.7741852	best: 0.7741852 (0)	total: 246ms	remaining: 20m 29s
200:	test: 0.9360722	best: 0.9360722 (200)	total: 38.4s	remaining: 15m 16s
400:	test: 0.9421387	best: 0.9421387 (400)	total: 1m 16s	remaining: 14m 35s
600:	test: 0.9435670	best: 0.9440513 (522)	total: 1m 54s	remaining: 13m 56s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9444026591
bestIteration = 657

Shrink model to first 658 iterations.
iter 1
0:	test: 0.7660201	best: 0.7660201 (0)	total: 223ms	remaining: 18m 33s
200:	test: 0.9274251	best: 0.9274251 (200)	total: 38.1s	remaining: 15m 10s
400:	test: 0.9348305	best: 0.9348305 (400)	total: 1m 16s	remaining: 14m 32s
600:	test: 0.9373677	best: 0.9374307 (599)	total: 1m 54s	remaining: 13m 54s
800:	test: 0.9393738	best: 0.9394445 (795)	total: 2m 31s	remaining: 13m 16s
1000:	test: 0.9401705	best: 0.9403520 (917)	total: 3m 9s	remaining: 12m 37s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9403520034
bestIteration = 917

## Обучаю модель на лучших параметрах и полных данных

In [65]:
%%time
vectorizer = TfidfVectorizer(analyzer='char_wb',
                             ngram_range=clf.best_params_['vector__ngram_range'],
                             max_df=clf.best_params_['vector__max_df'],
                             min_df=clf.best_params_['vector__min_df'],
                            )
vectorizer.fit(df.text_cl)
train = vectorizer.transform(df.text_cl)

model = CatBoostClassifier(iterations=int(np.mean(iterations)),
                           learning_rate=clf.best_params_['model__learning_rate'],
                           depth=clf.best_params_['model__depth'],
                           l2_leaf_reg=clf.best_params_['model__l2_leaf_reg'],
                           verbose=100,
                           )
model.fit(train, df.target)

0:	learn: 0.6714095	total: 226ms	remaining: 3m 1s
100:	learn: 0.3090736	total: 19.8s	remaining: 2m 17s
200:	learn: 0.2012872	total: 39.3s	remaining: 1m 57s
300:	learn: 0.1331894	total: 58.9s	remaining: 1m 38s
400:	learn: 0.0930633	total: 1m 18s	remaining: 1m 18s
500:	learn: 0.0670072	total: 1m 38s	remaining: 59.1s
600:	learn: 0.0495718	total: 1m 57s	remaining: 39.5s
700:	learn: 0.0378363	total: 2m 17s	remaining: 20s
800:	learn: 0.0292376	total: 2m 37s	remaining: 393ms
802:	learn: 0.0290419	total: 2m 37s	remaining: 0us
CPU times: user 18min 58s, sys: 35.6 s, total: 19min 34s
Wall time: 2min 49s


## Сохраняю

In [71]:
with open(os.path.join(PATH_MODL, 'catboost_model.pkl'), 'wb') as fd:
    pkl.dump(model, fd)
    
with open(os.path.join(PATH_MODL, 'catboost_vektorizer.pkl'), 'wb') as fd:
    pkl.dump(vectorizer, fd)